# Latihan Praktikum Bab 4: Feature Selection dan Pipeline
Jalankan dari atas ke bawah (**Shift+Enter** pada tiap sel), atau **Runtime → Run all**.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, RFE, SelectFromModel
from sklearn.metrics import accuracy_score, f1_score

---
# Praktikum 1: Membandingkan Feature Selection
Dataset pilihan: **Telco Customer Churn** (klasifikasi, 30 fitur setelah one-hot encoding). Tiga metode:

1. **SelectKBest** (Filter): memilih 10 fitur dengan skor uji statistik (ANOVA F) tertinggi.
2. **RFE** (Wrapper): membuang fitur paling tidak penting secara bertahap sampai tersisa 10, memakai Logistic Regression.
3. **Lasso / regularisasi L1** (Embedded): model sendiri yang menolkan koefisien fitur yang tidak berguna, jadi jumlah fitur ditentukan model.

Seleksi fitur hanya dipelajari dari **data latih** supaya tidak bocor ke data uji.

In [2]:
import os
# Baca dari folder data/ jika ada (repositori lokal); jika tidak (misalnya di Colab), ambil dari internet
lokal = "../data/Telco-Customer-Churn.csv"
url = lokal if os.path.exists(lokal) else "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna().drop(columns="customerID")
df["Churn"] = (df["Churn"] == "Yes").astype(int)
df = pd.get_dummies(df, drop_first=True)

X = df.drop(columns="Churn")
y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X.columns)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X.columns)
print("Jumlah fitur awal:", X.shape[1])

Jumlah fitur awal: 30


In [3]:
# Praktikum 4.3
K = 10

# 1) Filter: SelectKBest
skb = SelectKBest(score_func=f_classif, k=K).fit(X_train_s, y_train)

# 2) Wrapper: RFE
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=K).fit(X_train_s, y_train)

# 3) Embedded: Lasso (regularisasi L1)
lasso = SelectFromModel(
    LogisticRegression(penalty="l1", solver="liblinear", C=0.01, random_state=42)
).fit(X_train_s, y_train)

terpilih = {
    "SelectKBest": set(X.columns[skb.get_support()]),
    "RFE": set(X.columns[rfe.get_support()]),
    "Lasso": set(X.columns[lasso.get_support()]),
}
for nama, fitur in terpilih.items():
    print(f"\n{nama} ({len(fitur)} fitur):")
    print(sorted(fitur))


SelectKBest (10 fitur):
['Contract_Two year', 'DeviceProtection_No internet service', 'InternetService_Fiber optic', 'OnlineBackup_No internet service', 'OnlineSecurity_No internet service', 'PaymentMethod_Electronic check', 'StreamingMovies_No internet service', 'StreamingTV_No internet service', 'TechSupport_No internet service', 'tenure']

RFE (10 fitur):
['Contract_One year', 'Contract_Two year', 'InternetService_Fiber optic', 'MonthlyCharges', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'TotalCharges', 'tenure']

Lasso (15 fitur):
['Contract_One year', 'Contract_Two year', 'Dependents_Yes', 'InternetService_Fiber optic', 'OnlineBackup_No internet service', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'PaperlessBilling_Yes', 'PaymentMethod_Electronic check', 'SeniorCitizen', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'StreamingTV_Yes', 'TechSupport_Yes', 'tenure']


### Perbandingan fitur yang terpilih

In [4]:
# Tabel: fitur mana dipilih oleh metode mana
tabel = pd.DataFrame({nama: [("✓" if f in fitur else "") for f in X.columns]
                      for nama, fitur in terpilih.items()}, index=X.columns)
tabel["Jumlah metode"] = (tabel != "").sum(axis=1)
tabel = tabel[tabel["Jumlah metode"] > 0].sort_values("Jumlah metode", ascending=False)
print("Dipilih oleh ketiga metode:", sorted(tabel[tabel["Jumlah metode"] == 3].index))
tabel

Dipilih oleh ketiga metode: ['Contract_Two year', 'InternetService_Fiber optic', 'StreamingMovies_No internet service', 'tenure']


,SelectKBest,RFE,Lasso,Jumlah metode
tenure,✓,✓,✓,3
Contract_Two year,✓,✓,✓,3
StreamingMovies_No internet service,✓,✓,✓,3
InternetService_Fiber optic,✓,✓,✓,3
PaymentMethod_Electronic check,✓,,✓,2
Contract_One year,,✓,✓,2
OnlineBackup_No internet service,✓,,✓,2
OnlineSecurity_No internet service,✓,,✓,2
StreamingTV_No internet service,✓,✓,,2
StreamingTV_Yes,,✓,✓,2


In [5]:
# Seberapa mirip pilihan antar metode (Jaccard)
nama = list(terpilih)
for i in range(3):
    for j in range(i + 1, 3):
        a, b = terpilih[nama[i]], terpilih[nama[j]]
        print(f"{nama[i]} vs {nama[j]}: irisan={len(a & b)}, Jaccard={len(a & b) / len(a | b):.2f}")

SelectKBest vs RFE: irisan=5, Jaccard=0.33
SelectKBest vs Lasso: irisan=7, Jaccard=0.39
RFE vs Lasso: irisan=7, Jaccard=0.39


In [6]:
# Apakah performa model tetap baik dengan fitur yang lebih sedikit?
hasil = []
semua = {"Semua fitur": list(X.columns), **{k: sorted(v) for k, v in terpilih.items()}}
for nama, kolom in semua.items():
    m = LogisticRegression(max_iter=1000).fit(X_train_s[kolom], y_train)
    pred = m.predict(X_test_s[kolom])
    hasil.append({"Metode": nama, "Jumlah fitur": len(kolom),
                  "Akurasi": round(accuracy_score(y_test, pred), 4),
                  "F1 (churn)": round(f1_score(y_test, pred), 4)})
pd.DataFrame(hasil)

,Metode,Jumlah fitur,Akurasi,F1 (churn)
0,Semua fitur,30,0.8038,0.6091
1,SelectKBest,10,0.7925,0.5549
2,RFE,10,0.7868,0.5665
3,Lasso,15,0.7918,0.5796


**Kesimpulan Praktikum 1 (isi/sesuaikan dengan output Anda):**
- Ketiga metode memilih fitur yang mirip tetapi tidak identik (kemiripan Jaccard hanya 0,33 sampai 0,39). Empat fitur dipilih oleh ketiganya: `tenure`, `Contract_Two year`, `InternetService_Fiber optic`, dan `StreamingMovies_No internet service`. Ini masuk akal secara bisnis: pelanggan baru, tanpa kontrak panjang, dan pengguna fiber optic lebih rentan churn.
- Perhatikan bahwa semua kolom `..._No internet service` isinya identik (menandai pelanggan tanpa internet). SelectKBest menilai tiap fitur sendiri-sendiri, sehingga memakai 6 dari 10 slotnya untuk informasi yang sama. Ini kelemahan metode Filter, sedangkan RFE dan Lasso lebih beragam.
- Dengan 10 sampai 15 fitur dari 30, akurasi turun tipis (0,787 sampai 0,793 dibanding 0,804) dan F1 turun sekitar 0,03 sampai 0,05. Jadi model lebih sederhana dengan sedikit pengorbanan performa.
- **Filter** paling cepat. **Wrapper (RFE)** mempertimbangkan model sehingga lebih akurat tetapi lebih lambat. **Embedded (Lasso)** seleksi terjadi saat pelatihan dan jumlah fitur ditentukan otomatis (di sini 15).

---
# Praktikum 2: Membangun Pipeline Lengkap (Titanic)
ColumnTransformer untuk kolom campuran:
- numerik (`age`, `fare`): imputasi median lalu standarisasi
- kategorikal (`sex`, `embarked`): imputasi modus lalu one-hot encoding

Semuanya dibungkus bersama model klasifikasi dalam satu Pipeline.

In [7]:
titanic = sns.load_dataset("titanic")
print(titanic.shape)
print(titanic[["age", "fare", "sex", "embarked"]].isna().sum())

fitur_num = ["age", "fare"]
fitur_kat = ["sex", "embarked"]
Xt = titanic[fitur_num + fitur_kat]
yt = titanic["survived"]

Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    Xt, yt, test_size=0.2, stratify=yt, random_state=42
)

(891, 15)
age         177
fare          0
sex           0
embarked      2
dtype: int64


In [8]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), fitur_num),
    ("kat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), fitur_kat),
])

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])

pipeline.fit(Xt_train, yt_train)
pred = pipeline.predict(Xt_test)
print("Akurasi data uji (Logistic Regression):", round(accuracy_score(yt_test, pred), 4))
print("Akurasi cross-validation 5-fold:",
      round(cross_val_score(pipeline, Xt_train, yt_train, cv=5).mean(), 4))

Akurasi data uji (Logistic Regression): 0.7765
Akurasi cross-validation 5-fold: 0.7824


In [9]:
# Pembanding: ganti model dengan Random Forest (preprocessing tidak perlu ditulis ulang)
pipeline_rf = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42)),
])
pipeline_rf.fit(Xt_train, yt_train)
print("Akurasi data uji (Random Forest):",
      round(accuracy_score(yt_test, pipeline_rf.predict(Xt_test)), 4))

Akurasi data uji (Random Forest): 0.7821


In [10]:
# Bukti manfaat Pipeline (bagian 4.15): prediksi data baru yang mentah, ada nilai kosong
data_baru = pd.DataFrame({
    "age": [22, np.nan],
    "fare": [7.25, 80.0],
    "sex": ["male", "female"],
    "embarked": ["S", np.nan],
})
print(pipeline.predict(data_baru))   # 1 = selamat, 0 = tidak

[0 1]


**Kesimpulan Praktikum 2 (isi/sesuaikan dengan output Anda):** Pipeline berhasil menangani kolom campuran dan nilai kosong (`age`, `embarked`) dalam satu objek. Akurasi sekitar 0,78 hanya dengan 4 fitur (Logistic Regression 0,777; Random Forest 0,782). Data baru yang mentah bisa langsung diprediksi dengan `pipeline.predict(...)` tanpa mengulang preprocessing, sehingga risiko kesalahan berkurang dan deployment lebih sederhana.